# Embedded C — Part 2: Drivers, State Machines and Interrupts

Part 1 was about single registers and bits. This part is about **structuring firmware**: how professional embedded code is split into layers and modules, how it reacts to events, and how interrupts hand data to the main program safely.

## What you'll build

```
/content/embedded_c/part2/
├── led_button/            a layered, modular driver project
│   ├── include/
│   │   ├── hal_gpio.h     Hardware Abstraction Layer interface
│   │   ├── led.h          LED driver
│   │   └── button.h       debounced button driver
│   ├── src/
│   │   ├── hal_gpio_sim.c HAL for the PC simulation
│   │   ├── led.c
│   │   ├── button.c
│   │   └── main.c         application: super loop
│   ├── port/
│   │   └── hal_gpio_stm32.c  the same HAL for a real STM32 (reference)
│   └── Makefile
├── state_machine/
│   └── traffic_light.c    table-driven finite state machine
├── uart_ring_buffer/
│   ├── include/ring_buffer.h
│   ├── src/ring_buffer.c  lock-free single-producer / single-consumer FIFO
│   ├── src/main.c         simulated UART interrupt + command parser
│   └── Makefile
└── timer_interrupt/
    └── systick_sim.c      a real asynchronous "interrupt" using Linux signals
```

## 0. Setup

In [ ]:
!mkdir -p /content/embedded_c/part2/led_button/include /content/embedded_c/part2/led_button/src /content/embedded_c/part2/led_button/port
!mkdir -p /content/embedded_c/part2/state_machine /content/embedded_c/part2/uart_ring_buffer/include /content/embedded_c/part2/uart_ring_buffer/src /content/embedded_c/part2/timer_interrupt
!echo "ready" && ls /content/embedded_c/part2

---
## 1. Layered firmware: HAL → drivers → application

If your application code writes `GPIOA->ODR |= ...` directly, it's welded to one chip. Professional firmware stacks thin layers, each talking only to the one below:

```
 ┌──────────────────────────────┐
 │ Application   (main.c)       │  "when the button is pressed, toggle the LED"
 ├──────────────────────────────┤
 │ Drivers       (led.c, button.c) │  LED on/off, active-low, debouncing
 ├──────────────────────────────┤
 │ HAL           (hal_gpio.h)   │  hal_gpio_write(pin, level)
 ├──────────────┬───────────────┤
 │ hal_gpio_sim.c│ hal_gpio_stm32.c │  one implementation per platform
 └──────────────┴───────────────┘
```

To move to a new microcontroller you rewrite **only the HAL `.c` file**; the drivers and application compile unchanged. It also lets you test driver logic on a PC, which is exactly what we're doing here.

Two ideas to notice in the drivers:

- **Driver state lives in a struct** (`led_t`, `button_t`), and functions take a pointer to it. That's how one driver serves many LEDs or buttons: object-oriented design in plain C.
- Everything a module doesn't need to expose is `static`.

In [ ]:
%%writefile /content/embedded_c/part2/led_button/include/hal_gpio.h
#ifndef HAL_GPIO_H
#define HAL_GPIO_H

#include <stdint.h>
#include <stdbool.h>

typedef enum { HAL_PIN_INPUT, HAL_PIN_OUTPUT } hal_pin_mode_t;

/* The only GPIO functions the drivers are allowed to use */
void hal_gpio_init(uint8_t pin, hal_pin_mode_t mode);
void hal_gpio_write(uint8_t pin, bool level);
bool hal_gpio_read(uint8_t pin);

/* Simulation-only helpers: these do not exist on real hardware */
void     sim_set_input(uint8_t pin, bool level);
uint32_t sim_get_output_register(void);

#endif /* HAL_GPIO_H */

In [ ]:
%%writefile /content/embedded_c/part2/led_button/src/hal_gpio_sim.c
/* HAL implementation for the PC: "registers" are plain variables. */
#include "hal_gpio.h"

static uint32_t mode_reg;     /* bit = 1 means the pin is an output */
static uint32_t output_reg;
static uint32_t input_reg;

void hal_gpio_init(uint8_t pin, hal_pin_mode_t mode)
{
    if (mode == HAL_PIN_OUTPUT) mode_reg |=  (1u << pin);
    else                        mode_reg &= ~(1u << pin);
}

void hal_gpio_write(uint8_t pin, bool level)
{
    if (!(mode_reg & (1u << pin))) return;    /* writes to input pins are ignored */
    if (level) output_reg |=  (1u << pin);
    else       output_reg &= ~(1u << pin);
}

bool hal_gpio_read(uint8_t pin)
{
    return (input_reg >> pin) & 1u;
}

void sim_set_input(uint8_t pin, bool level)
{
    if (level) input_reg |=  (1u << pin);
    else       input_reg &= ~(1u << pin);
}

uint32_t sim_get_output_register(void)
{
    return output_reg;
}

In [ ]:
%%writefile /content/embedded_c/part2/led_button/port/hal_gpio_stm32.c
/* The SAME HAL interface implemented for a real STM32F4 (GPIOA only, for illustration).
   Not built in Colab. On a board you'd compile this file instead of hal_gpio_sim.c,
   and led.c / button.c would not change at all. */
#include "hal_gpio.h"

#define RCC_AHB1ENR  (*(volatile uint32_t *)0x40023830u)
#define GPIOA_MODER  (*(volatile uint32_t *)0x40020000u)
#define GPIOA_IDR    (*(volatile uint32_t *)0x40020010u)
#define GPIOA_BSRR   (*(volatile uint32_t *)0x40020018u)

void hal_gpio_init(uint8_t pin, hal_pin_mode_t mode)
{
    RCC_AHB1ENR |= 1u;                                /* clock for GPIOA */
    GPIOA_MODER &= ~(3u << (pin * 2));
    if (mode == HAL_PIN_OUTPUT) GPIOA_MODER |= (1u << (pin * 2));
}

void hal_gpio_write(uint8_t pin, bool level)
{
    GPIOA_BSRR = level ? (1u << pin) : (1u << (pin + 16));   /* atomic, no read-modify-write */
}

bool hal_gpio_read(uint8_t pin)
{
    return (GPIOA_IDR >> pin) & 1u;
}

In [ ]:
%%writefile /content/embedded_c/part2/led_button/include/led.h
#ifndef LED_H
#define LED_H

#include <stdint.h>
#include <stdbool.h>

typedef struct {
    uint8_t pin;
    bool    active_low;   /* many boards light the LED by driving the pin LOW */
    bool    is_on;
} led_t;

void led_init(led_t *led, uint8_t pin, bool active_low);
void led_on(led_t *led);
void led_off(led_t *led);
void led_toggle(led_t *led);

#endif /* LED_H */

In [ ]:
%%writefile /content/embedded_c/part2/led_button/src/led.c
#include "led.h"
#include "hal_gpio.h"

static void apply(const led_t *led)          /* private helper */
{
    bool level = led->active_low ? !led->is_on : led->is_on;
    hal_gpio_write(led->pin, level);
}

void led_init(led_t *led, uint8_t pin, bool active_low)
{
    led->pin        = pin;
    led->active_low = active_low;
    led->is_on      = false;
    hal_gpio_init(pin, HAL_PIN_OUTPUT);
    apply(led);
}

void led_on(led_t *led)     { led->is_on = true;        apply(led); }
void led_off(led_t *led)    { led->is_on = false;       apply(led); }
void led_toggle(led_t *led) { led->is_on = !led->is_on; apply(led); }

### Debouncing

A mechanical button doesn't switch cleanly. For a few milliseconds its contacts **bounce**, flickering between 0 and 1. Read it naively and one press can count as five.

The driver below accepts a new level only after the raw signal has stayed the same for `DEBOUNCE_MS`. It never uses `delay()`: it's called on every pass of the main loop with the current time, does a tiny bit of work, and returns. That **non-blocking** style keeps the rest of the firmware responsive.

In [ ]:
%%writefile /content/embedded_c/part2/led_button/include/button.h
#ifndef BUTTON_H
#define BUTTON_H

#include <stdint.h>
#include <stdbool.h>

#define DEBOUNCE_MS 20u

typedef struct {
    uint8_t  pin;
    bool     stable_state;     /* the debounced level */
    bool     last_raw;         /* the most recent raw reading */
    uint32_t last_change_ms;   /* when the raw reading last changed */
} button_t;

void button_init(button_t *b, uint8_t pin);

/* Call often (every main-loop pass). Returns true exactly once per clean press. */
bool button_update(button_t *b, uint32_t now_ms);

#endif /* BUTTON_H */

In [ ]:
%%writefile /content/embedded_c/part2/led_button/src/button.c
#include "button.h"
#include "hal_gpio.h"

void button_init(button_t *b, uint8_t pin)
{
    b->pin = pin;
    hal_gpio_init(pin, HAL_PIN_INPUT);
    b->last_raw = b->stable_state = hal_gpio_read(pin);
    b->last_change_ms = 0;
}

bool button_update(button_t *b, uint32_t now_ms)
{
    bool raw = hal_gpio_read(b->pin);

    if (raw != b->last_raw) {              /* still bouncing: restart the timer */
        b->last_raw       = raw;
        b->last_change_ms = now_ms;
        return false;
    }

    /* unsigned subtraction stays correct even when the ms counter wraps around */
    if ((now_ms - b->last_change_ms) >= DEBOUNCE_MS && raw != b->stable_state) {
        b->stable_state = raw;             /* steady long enough: accept it */
        return raw;                        /* report presses (0 -> 1), not releases */
    }
    return false;
}

### The application: a super loop

`main()` initialises the drivers and then loops forever. Each pass, it polls the drivers and reacts. On a real board the loop would run as fast as possible, and `now` would come from a hardware timer (section 4). Here, one pass is one simulated millisecond, and a script feeds in a bouncy button signal.

In [ ]:
%%writefile /content/embedded_c/part2/led_button/src/main.c
#include <stdio.h>
#include <stddef.h>
#include "hal_gpio.h"
#include "led.h"
#include "button.h"

#define LED_PIN    5
#define BUTTON_PIN 0

/* A pretend button signal: (time in ms, level). Quick flips are contact bounce. */
typedef struct { uint32_t t; bool level; } sample_t;

static const sample_t input_script[] = {
    {100, 1}, {102, 0}, {104, 1}, {105, 0}, {107, 1},   /* bouncy press      */
    {300, 0}, {302, 1}, {303, 0},                       /* bouncy release    */
    {500, 1},                                           /* clean press       */
    {700, 0},                                           /* release           */
    {800, 1}, {805, 0},                                 /* 5 ms noise spike  */
};
#define SCRIPT_LEN (sizeof input_script / sizeof input_script[0])

int main(void)
{
    led_t    led;
    button_t btn;
    led_init(&led, LED_PIN, false);
    button_init(&btn, BUTTON_PIN);

    size_t next = 0;
    int presses = 0;

    for (uint32_t now = 0; now <= 1000; now++) {      /* super loop: 1 pass = 1 ms */
        while (next < SCRIPT_LEN && input_script[next].t == now) {
            sim_set_input(BUTTON_PIN, input_script[next].level);   /* "the world" */
            next++;
        }

        if (button_update(&btn, now)) {
            presses++;
            led_toggle(&led);
            printf("[%4u ms] clean press #%d -> LED %s  (output register = 0x%02X)\n",
                   (unsigned)now, presses, led.is_on ? "ON " : "OFF",
                   (unsigned)sim_get_output_register());
        }
    }
    printf("\n12 raw transitions in the input, %d real presses detected.\n", presses);
    return 0;
}

In [ ]:
%%writefile /content/embedded_c/part2/led_button/Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -std=c11 -Iinclude
SRC     = $(wildcard src/*.c)
OBJ     = $(patsubst src/%.c,build/%.o,$(SRC))
TARGET  = build/led_button

all: $(TARGET)

$(TARGET): $(OBJ)
	$(CC) $(OBJ) -o $@

build/%.o: src/%.c include/*.h | build
	$(CC) $(CFLAGS) -c $< -o $@

build:
	mkdir -p build

run: $(TARGET)
	./$(TARGET)

clean:
	rm -rf build

.PHONY: all run clean

In [ ]:
!cd /content/embedded_c/part2/led_button && make clean && make && echo && make -s run

The press at 100 ms bounces until 107 ms and is accepted 20 ms later at 127 ms. The bouncy release and the 5 ms spike at 800 ms are both filtered out.

Try changing `DEBOUNCE_MS` to `3` in `button.h`, then re-run the header cell and `make`: the bounces start counting as presses. Because the Makefile lists `include/*.h` as a dependency, editing a header rebuilds everything that uses it.

---
## 2. Finite state machines

Most embedded devices are state machines: a washing machine is *filling → washing → rinsing → spinning*, and a protocol parser is *waiting for header → reading length → reading payload*. Writing them explicitly, instead of as a tangle of flags, makes behaviour predictable and easy to test.

The pattern:

- an `enum` of **states**,
- **events** (timeouts, button presses, received bytes),
- **transitions**: in state X, on event Y, go to state Z (and maybe do an action).

The traffic light below is **table-driven**: each state's name, duration and normal next state live in a `const` array, so adding a state is mostly a table edit. A `switch` handles the special event, a pedestrian button that cuts GREEN short, but only after a minimum green time.

In [ ]:
%%writefile /content/embedded_c/part2/state_machine/traffic_light.c
#include <stdio.h>
#include <stdint.h>
#include <stddef.h>

typedef enum { ST_RED, ST_RED_AMBER, ST_GREEN, ST_AMBER, ST_COUNT } state_t;
typedef enum { EV_PEDESTRIAN } event_t;

typedef struct {
    const char *name;
    uint32_t    duration_s;    /* how long to stay before the timeout transition */
    state_t     on_timeout;    /* where to go when time is up */
} state_info_t;

/* const tables end up in Flash on a microcontroller, not in scarce RAM */
static const state_info_t states[ST_COUNT] = {
    [ST_RED]       = { "RED",       5, ST_RED_AMBER },
    [ST_RED_AMBER] = { "RED+AMBER", 1, ST_GREEN     },
    [ST_GREEN]     = { "GREEN",     6, ST_AMBER     },
    [ST_AMBER]     = { "AMBER",     2, ST_RED       },
};

#define MIN_GREEN_S 2u

static state_t  current;
static uint32_t time_in_state;

static void enter(state_t s, uint32_t now)
{
    current = s;
    time_in_state = 0;
    printf("[t=%2u s] -> %s\n", (unsigned)now, states[s].name);
}

static void handle_event(event_t ev, uint32_t now)
{
    switch (current) {
    case ST_GREEN:
        if (ev == EV_PEDESTRIAN && time_in_state >= MIN_GREEN_S) {
            printf("[t=%2u s]    pedestrian button: cutting GREEN short\n", (unsigned)now);
            enter(ST_AMBER, now);
        }
        break;
    default:
        printf("[t=%2u s]    pedestrian button ignored (light is not GREEN)\n", (unsigned)now);
        break;
    }
}

int main(void)
{
    const uint32_t button_presses[] = { 3, 9 };      /* seconds at which someone presses */
    const size_t   n_presses = sizeof button_presses / sizeof button_presses[0];
    size_t next = 0;

    enter(ST_RED, 0);
    for (uint32_t now = 1; now <= 26; now++) {      /* one tick = one second */
        time_in_state++;

        if (next < n_presses && button_presses[next] == now) {
            handle_event(EV_PEDESTRIAN, now);
            next++;
        }
        if (time_in_state >= states[current].duration_s)
            enter(states[current].on_timeout, now);
    }
    return 0;
}

In [ ]:
!cd /content/embedded_c/part2/state_machine && gcc -Wall -Wextra -std=c11 traffic_light.c -o traffic_light && ./traffic_light

`[ST_GREEN] = {...}` is a C99 **designated initialiser**: each table row is tied to its enum value, so reordering the enum can't silently mismatch the table.

---
## 3. Interrupts and the ring buffer

An **interrupt** is the hardware pausing your main loop to run a short function, an **ISR** (Interrupt Service Routine), the moment something happens: a byte arrived on the UART, a timer expired, a pin changed.

Golden rules for ISRs:

1. **Be short.** Grab the data, stash it, return. No `printf`, no waiting, no `malloc`.
2. **Share data safely.** Shared variables must be `volatile`, and you must think about what happens if the ISR fires halfway through the main code's update.

The standard tool for passing a stream of bytes from an ISR to the main loop is a **ring buffer** (circular FIFO):

```
   data:  [ . . H E L L O . . . . . . . . . ]
                ^tail      ^head
   ISR writes at head, then head++       (only the ISR touches head)
   main reads at tail, then tail++       (only main touches tail)
```

Because each index has exactly **one writer**, this single-producer / single-consumer design needs no locks. The size is a power of two, so `index & (SIZE-1)` replaces the slower `%`, and the free-running 16-bit counters can even wrap around safely.

In [ ]:
%%writefile /content/embedded_c/part2/uart_ring_buffer/include/ring_buffer.h
#ifndef RING_BUFFER_H
#define RING_BUFFER_H

#include <stdint.h>
#include <stdbool.h>

#define RB_SIZE 16u                 /* must be a power of two */
#define RB_MASK (RB_SIZE - 1u)

typedef struct {
    uint8_t           data[RB_SIZE];
    volatile uint16_t head;         /* written only by the producer (the ISR)  */
    volatile uint16_t tail;         /* written only by the consumer (main loop) */
} ring_buffer_t;

void     rb_init(ring_buffer_t *rb);
bool     rb_put(ring_buffer_t *rb, uint8_t byte);    /* false if full  */
bool     rb_get(ring_buffer_t *rb, uint8_t *byte);   /* false if empty */
uint16_t rb_count(const ring_buffer_t *rb);
bool     rb_is_empty(const ring_buffer_t *rb);
bool     rb_is_full(const ring_buffer_t *rb);

#endif /* RING_BUFFER_H */

In [ ]:
%%writefile /content/embedded_c/part2/uart_ring_buffer/src/ring_buffer.c
#include "ring_buffer.h"

void rb_init(ring_buffer_t *rb)
{
    rb->head = 0;
    rb->tail = 0;
}

uint16_t rb_count(const ring_buffer_t *rb)
{
    return (uint16_t)(rb->head - rb->tail);    /* correct even after the counters wrap */
}

bool rb_is_empty(const ring_buffer_t *rb) { return rb_count(rb) == 0; }
bool rb_is_full(const ring_buffer_t *rb)  { return rb_count(rb) >= RB_SIZE; }

bool rb_put(ring_buffer_t *rb, uint8_t byte)
{
    if (rb_is_full(rb)) return false;
    rb->data[rb->head & RB_MASK] = byte;       /* & RB_MASK is a fast % RB_SIZE */
    rb->head++;                                /* publish only after the data is stored */
    return true;
}

bool rb_get(ring_buffer_t *rb, uint8_t *byte)
{
    if (rb_is_empty(rb)) return false;
    *byte = rb->data[rb->tail & RB_MASK];
    rb->tail++;
    return true;
}

In [ ]:
%%writefile /content/embedded_c/part2/uart_ring_buffer/src/main.c
#include <stdio.h>
#include <string.h>
#include "ring_buffer.h"

static ring_buffer_t uart_rx;
static volatile unsigned dropped = 0;

/* On a real MCU the hardware calls this whenever a byte arrives on the UART. */
static void UART_RX_IRQHandler(uint8_t received)
{
    if (!rb_put(&uart_rx, received)) dropped++;    /* never wait inside an ISR */
}

static void process_command(const char *cmd)
{
    if      (strcmp(cmd, "LED ON")  == 0) printf("      >> turning LED on\n");
    else if (strcmp(cmd, "LED OFF") == 0) printf("      >> turning LED off\n");
    else if (strcmp(cmd, "STATUS")  == 0) printf("      >> status OK, dropped bytes = %u\n", dropped);
    else                                  printf("      >> unknown command \"%s\"\n", cmd);
}

int main(void)
{
    rb_init(&uart_rx);
    const char *incoming = "LED ON\nSTATUS\nHELLO\nLED OFF\n";
    size_t pos = 0, len = strlen(incoming);

    char   line[32];
    size_t line_len = 0;

    for (int pass = 0; pos < len || !rb_is_empty(&uart_rx); pass++) {
        /* 1) "Interrupts": up to 5 bytes arrive while the main loop is busy */
        for (int i = 0; i < 5 && pos < len; i++)
            UART_RX_IRQHandler((uint8_t)incoming[pos++]);
        printf("pass %2d: %2u byte(s) buffered\n", pass, rb_count(&uart_rx));

        /* 2) Main loop handles at most 3 bytes per pass (it has other work too) */
        uint8_t byte;
        for (int n = 0; n < 3 && rb_get(&uart_rx, &byte); n++) {
            if (byte == '\n') {
                line[line_len] = '\0';
                printf("    got line \"%s\"\n", line);
                process_command(line);
                line_len = 0;
            } else if (line_len < sizeof line - 1) {
                line[line_len++] = (char)byte;
            }
        }
    }

    /* Overflow test: 20 bytes arrive into a 16-byte buffer and nobody reads them */
    for (int i = 0; i < 20; i++) UART_RX_IRQHandler('x');
    printf("\noverflow test: buffer holds %u, dropped %u (no crash, no corruption)\n",
           rb_count(&uart_rx), dropped);
    return 0;
}

In [ ]:
%%writefile /content/embedded_c/part2/uart_ring_buffer/Makefile
CC      = gcc
CFLAGS  = -Wall -Wextra -std=c11 -Iinclude
SRC     = $(wildcard src/*.c)
OBJ     = $(patsubst src/%.c,build/%.o,$(SRC))
TARGET  = build/uart_demo

all: $(TARGET)

$(TARGET): $(OBJ)
	$(CC) $(OBJ) -o $@

build/%.o: src/%.c include/*.h | build
	$(CC) $(CFLAGS) -c $< -o $@

build:
	mkdir -p build

run: $(TARGET)
	./$(TARGET)

clean:
	rm -rf build

.PHONY: all run clean

In [ ]:
!cd /content/embedded_c/part2/uart_ring_buffer && make clean && make && echo && make -s run

Watch the buffered count climb, because bytes arrive faster (5 per pass) than the main loop drains them (3 per pass). The buffer absorbs the burst, and once input stops the loop catches up. If the burst outruns the buffer, bytes are dropped and counted rather than corrupting memory. On a real system you'd size `RB_SIZE` for the worst-case burst.

In this simulation the "ISR" is an ordinary function call, so it can't truly interrupt `rb_get`. The next section uses a genuinely asynchronous interrupt.

---
## 4. A real asynchronous "interrupt": simulating SysTick

Every Cortex-M chip has a **SysTick** timer that fires an interrupt at a fixed rate, typically every 1 ms. Its handler just increments a global tick counter, and the rest of the firmware uses that counter for timing: blinking, timeouts, debouncing.

Linux can imitate this: `setitimer` asks the kernel to send our program a `SIGALRM` signal every 10 ms, and a **signal handler** runs immediately, interrupting `main` wherever it is. That's the same situation as a hardware ISR, so the same rules apply:

- The shared counter is `volatile sig_atomic_t`, a type that can be read and written in one indivisible step.
- The handler does one tiny thing and returns.
- `main` copies the shared value **once** per pass into a local `now`, so all decisions in that pass see a consistent time.
- `pause()` sleeps until the next interrupt, like the ARM instruction `__WFI()` (wait for interrupt), which saves power on battery devices.

Timing is **non-blocking**: rather than `delay(250)`, the loop asks "have 250 ms passed since I last toggled?". The CPU stays free for other tasks, shown by the second, independent 500 ms job.

In [ ]:
%%writefile /content/embedded_c/part2/timer_interrupt/systick_sim.c
#define _POSIX_C_SOURCE 200809L      /* enables sigaction, setitimer in strict C11 mode */
#include <stdio.h>
#include <signal.h>
#include <sys/time.h>
#include <unistd.h>                  /* pause() */

static volatile sig_atomic_t tick_ms = 0;     /* shared between the "ISR" and main */

/* Plays the role of SysTick_Handler() on an ARM Cortex-M */
static void systick_handler(int sig)
{
    (void)sig;                       /* unused parameter */
    tick_ms += 10;                   /* our timer fires every 10 ms */
}

static void start_systick_10ms(void)
{
    struct sigaction sa = {0};
    sa.sa_handler = systick_handler;
    sigemptyset(&sa.sa_mask);
    sigaction(SIGALRM, &sa, NULL);   /* like enabling the interrupt in the NVIC */

    struct itimerval tv = {0};
    tv.it_interval.tv_usec = 10000;  /* period: 10 ms */
    tv.it_value.tv_usec    = 10000;  /* first tick after 10 ms */
    setitimer(ITIMER_REAL, &tv, NULL);
}

int main(void)
{
    start_systick_10ms();

    int led = 0, toggles = 0, reports = 0;
    int last_blink = 0, last_report = 0;

    while (toggles < 8) {                        /* the super loop */
        int now = tick_ms;                       /* read the shared value ONCE */

        if (now - last_blink >= 250) {           /* task 1: blink every 250 ms */
            last_blink = now;
            led = !led;
            toggles++;
            printf("[%5d ms] LED %s\n", now, led ? "ON" : "OFF");
        }

        if (now - last_report >= 500) {          /* task 2: independent 500 ms job */
            last_report = now;
            reports++;
            printf("[%5d ms]          sensor read #%d (runs alongside the blinking)\n", now, reports);
        }

        pause();                                 /* sleep until the next interrupt (__WFI) */
    }
    printf("\nfinished after %d ms of 'real' time\n", (int)tick_ms);
    return 0;
}

In [ ]:
!cd /content/embedded_c/part2/timer_interrupt && gcc -Wall -Wextra -std=c11 systick_sim.c -o systick_sim && ./systick_sim

The program really ran for about two seconds of wall-clock time, driven entirely by the timer interrupt. Times may drift by ±10 ms between runs: that's real asynchronous behaviour, not a scripted simulation.

### What changes on a real microcontroller

```c
volatile uint32_t tick_ms;

void SysTick_Handler(void)          // name fixed by the startup file's vector table
{
    tick_ms++;
}

int main(void)
{
    SysTick_Config(SystemCoreClock / 1000);   // CMSIS: interrupt every 1 ms
    for (;;) {
        uint32_t now = tick_ms;
        /* ... same non-blocking tasks as above ... */
        __WFI();
    }
}
```

On a 32-bit Cortex-M, reading an aligned `uint32_t` is a single instruction, so it's safe. On an **8-bit** AVR, reading a 32-bit counter takes four instructions and the ISR can fire in between. There you briefly disable interrupts around the read (`cli(); now = tick_ms; sei();`), a **critical section**.

---
# Exercises

1. **Driver:** add `led_blink(led_t *led, uint32_t now_ms, uint32_t period_ms)` to the LED driver, written non-blocking, and use it in `main.c` so the LED blinks while waiting for presses.
2. **Button:** extend `button_t` to detect a **long press** (held for over 1 s) and report it separately from a short press.
3. **State machine:** add a `ST_FLASHING_AMBER` night mode, entered when a `EV_NIGHT` event arrives, and leave it on `EV_DAY`.
4. **Ring buffer:** add `rb_peek()` (look at the next byte without removing it). Then make the buffer generic over element size so it can queue `int16_t` ADC samples.
5. **Protocol parser:** replace the line parser in `uart_ring_buffer/src/main.c` with a state machine for frames shaped like `0xAA <len> <payload...> <checksum>`.

**Next:** *Embedded C — Part 3: Memory, Fixed-Point Math and Reliability.*